# Experiements for Progress Report 4/10/19

In [ ]:
import parametrization, sparse_matrices, numpy as np, importlib
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
m = parametrization.Mesh("../examples/lilium.msh")
lg = parametrization.LocalGlobalParametrizer(m, parametrization.lscm(m))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

print(lg.energy())
lg.runIteration()
print(lg.energy())

rparam = parametrization.RegularizedParametrizerSVD(lg)
for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    opts = parametrization.NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = 2000
    opts.gradTol = 1e-10
    parametrization.benchmark_reset()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)
    parametrization.benchmark_report()

In [ ]:
with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 5e-6)

In [ ]:
PET = parametrization.RegularizedParametrizerSVD.EnergyType
list(map(rparam.energy, [PET.Fitting, PET.AlphaRegularization, PET.PhiRegularization]))

In [ ]:
visualization.visualize(lg)

In [ ]:
visualization.visualize(rparam)

In [ ]:
importlib.reload(visualization)
visualization.visualizeChannelOrientation(rparam, quiver=True)

In [ ]:
visualization.visualizeChannelOrientation(lg, quiver=True)

In [ ]:
importlib.reload(visualization)
visualization.singularValueHistogram(rparam)

In [ ]:
visualization.singularValueHistogram(parametrization.RegularizedParametrizerSVD(lg))

## Prohibit variation of the stretch factor

In [ ]:
rparam.alphaMin = np.pi / 2
with suppress_stdout(): optimize_rparam(rparam, 0, 1e-6)
visualization.visualize(rparam)

In [ ]:
importlib.reload(visualization)
visualization.visualizeChannelOrientation(rparam, quiver=True)

In [ ]:
visualization.singularValueHistogram(rparam)

## Singular value statistics (for report tables)

In [ ]:
tmp = parametrization.RegularizedParametrizerSVD(lg)
(np.min(tmp.getMinSingularValues()), np.max(tmp.getMinSingularValues())), (np.min(tmp.getAlphas()), np.max(tmp.getAlphas()))

In [ ]:
(np.min(rparam.getMinSingularValues()), np.max(rparam.getMinSingularValues())), (np.min(rparam.getAlphas()), np.max(rparam.getAlphas()))

## Read in the nice (but unreproducible) map found earlier

In [ ]:
rparam2 = parametrization.RegularizedParametrizerSVD(lg)
rparam2.setUV(np.loadtxt('data/lilium_tower_parametrization.txt'))
rparam2.alphaMin = 1.4
rparam2.alphaRegW = 1e-5
rparam2.phiRegW = 1e-5
visualization.visualize(rparam2)